# 【教員用】テーマ案からグループを自動編成する

情報活用　第1回（9/24・9/25）

**第1回の 64-77分でその場で使う。**フォームで集めたテーマ案を読み込み、**書かれている内容が近い人どうしで4人組**を作る。

- 付箋は使わない
- **同じクラスの中だけ**で組む
- 結果はそのまま投影できる形で出る
- **機械が決めるのは「たたき台」まで。**最後は目で見て動かす

In [ ]:
# 準備：▶ を押すだけ
import math, re, collections, pandas as pd
from google.colab import files
print("準備できました")

---
## 1. 回答のCSVを読み込む

1. テーマ案フォームの**集計スプレッドシート**を開く
2. ファイル → ダウンロード → **カンマ区切り形式(.csv)**
3. 下を実行して、そのファイルを選ぶ

In [ ]:
up = files.upload()
path = list(up.keys())[0]
df = pd.read_csv(path)
print(f"{len(df)} 件　列：")
for c in df.columns: print("  -", c)
df.head(3)

---
## 2. 使う列を決める

フォームの列名がそのまま入っている。**下の3つを、実際の列名に合わせる。**
（`df.columns` の一覧を見て、部分一致で自動的に拾うようにしてある。ずれていたら直す）

In [ ]:
COL_CLASS = [c for c in df.columns if "クラス" in c][0]
COL_NAME  = [c for c in df.columns if "氏名" in c][0]
COL_IDEAS = [c for c in df.columns if "案" in c]          # 案①〜案⑤
COL_WHY   = [c for c in df.columns if "なぜ" in c]

print("クラス :", COL_CLASS)
print("氏名   :", COL_NAME)
print("テーマ :", COL_IDEAS)

---
## 3. 似ているかどうかの測り方

**漢字・カタカナ・英数のかたまり**（学食／睡眠時間／バイト／SNS…）だけを取り出して比べる。

ひらがなの「〜のか」「〜している」「人ほど」は**どの文にも出てくる**ので、そこが一致しても近いことにはならない。**内容語だけを見る。**

さらに、語の中の2〜3文字も特徴に入れる。こうすると「**バイト**」と「アル**バイト**」、「睡眠時間」と「勉強時間」のような**部分的な重なり**も拾える。

In [ ]:
WORD = re.compile(r"[一-龥ヲ-ヴーァ-ヺA-Za-z0-9]+")

def content_words(text):
    return [w for w in WORD.findall(str(text)) if len(w) >= 2]

def features(text):
    out = []
    for w in content_words(text):
        out.append(w)
        for n in (2, 3):
            out += [w[i:i+n] for i in range(max(0, len(w)-n+1))]
    return out

def vectorize(docs):
    tf = [collections.Counter(features(d)) for d in docs]
    dfreq = collections.Counter()
    for c in tf:
        for k in c: dfreq[k] += 1
    N = len(docs); vecs = []
    for c in tf:
        v = {k: (1+math.log(f)) * math.log((N+1)/(dfreq[k]+1)) for k, f in c.items()}
        nrm = math.sqrt(sum(x*x for x in v.values())) or 1
        vecs.append({k: x/nrm for k, x in v.items()})
    return vecs

def cosine(a, b):
    if len(a) > len(b): a, b = b, a
    return sum(x * b.get(k, 0) for k, x in a.items())

print("定義しました")

---
## 4. グループを作る

**やり方**：いちばん似ている2人を核にして、その2人のどちらかと最も似ている人を足していく。4人になったら次の核へ。

**「誰とも似ていない人」は必ず出る。**その人は空いているグループに入る。出力に印を付けるので、**あとで手で動かす。**

In [ ]:
SIZE = 4   # 1グループの人数

def make_groups(docs, size=SIZE):
    vecs = vectorize(docs); n = len(docs)
    sim = [[cosine(vecs[i], vecs[j]) if i != j else -1 for j in range(n)] for i in range(n)]
    left = set(range(n)); groups = []
    while left:
        if len(left) <= size:
            groups.append(sorted(left)); break
        _, i, j = max(((sim[a][b], a, b) for a in left for b in left if a < b), key=lambda x: x[0])
        g = [i, j]; left -= {i, j}
        while len(g) < size and left:
            k = max(left, key=lambda k: max(sim[k][m] for m in g))
            g.append(k); left.discard(k)
        groups.append(g)
    return groups, sim

def common_words(docs, idx, top=3):
    cnt = collections.Counter()
    for i in idx:
        for w in set(content_words(docs[i])):
            if len(w) >= 2: cnt[w] += 1
    return [w for w, c in cnt.most_common(top) if c >= 2]

print("定義しました")

---
## 5. クラスを選んで実行する

**同じクラスの中だけで組む。**その日の授業のクラスを選ぶ。

In [ ]:
TARGET = df[COL_CLASS].dropna().unique()[0]   # 別のクラスにするときは書き換える
print("対象:", TARGET)

sub = df[df[COL_CLASS] == TARGET].drop_duplicates(subset=[COL_NAME], keep="last").reset_index(drop=True)
names = sub[COL_NAME].astype(str).tolist()
docs  = sub[COL_IDEAS].fillna("").agg(" ".join, axis=1).tolist()
first = sub[COL_IDEAS[0]].fillna("").astype(str).tolist()

groups, sim = make_groups(docs)

print(f"\n{TARGET}　{len(names)}名 → {len(groups)}グループ\n" + "=" * 58)
for gi, g in enumerate(groups, 1):
    kw = common_words(docs, g)
    print(f"\n【グループ{gi}】" + (f"　共通：{'・'.join(kw)}" if kw else "　共通：（なし）"))
    for i in g:
        near = max((sim[i][m] for m in g if m != i), default=0)
        mark = "　⚠" if near < 0.05 else "　"
        print(f"  {mark}{names[i]}　{first[i][:34]}")
print("\n" + "=" * 58)
print("⚠ ＝ 同じグループの誰とも内容が重なっていない人。手で動かす候補。")

---
## 6. 投影して、動かしてもらう

**そのまま確定にしない。**画面に出したうえで、こう言う。

> **これは機械が「書いてある言葉の近さ」だけで作ったものです。**
> 実際に話してみて、違うと思ったら動いて構いません。
> **⚠が付いている人は、近い人がいなかった人です。**気になるテーマのグループに移ってください。

そのうえで 69-77分の「集まる」時間をとる。**機械の出力は席替えの出発点であって、決定ではない。**

!!! note "最後のグループは寄せ集めになる"
    似ている人から順に組んでいくので、**最後に残った人たちが1グループになる。**
    ここは⚠だらけになりやすい。**そのグループから先に動いてもらう**とうまくいく。

---
## 7. 確定したら書き出す

手で動かしたあとの最終形を記録する。`グループ番号` の列を直してから実行する。

In [ ]:
rows = []
for gi, g in enumerate(groups, 1):
    for i in g:
        rows.append({"クラス": TARGET, "グループ": gi, "氏名": names[i], "テーマ案①": first[i]})
out = pd.DataFrame(rows)
out.to_csv("groups.csv", index=False, encoding="utf-8-sig")
files.download("groups.csv")
out

!!! warning "このファイルの扱い"
    `groups.csv` には**学生の氏名**が入る。**授業サイトのリポジトリには置かない。**
    手元か、大学の管理下（Moodle・OneDrive）に置く。